## ClinicalBERT Text Embedding (24h Nursing Notes → 768-D ICU-Stay Vectors)

In [ ]:
!pip -q install transformers torch accelerate

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

In [ ]:
OUT_DIR = "/content/drive/MyDrive/mimic_outputs"
NOTES_PATH = os.path.join(OUT_DIR, "notes_24h_nursing.parquet")
EMB_OUT = os.path.join(OUT_DIR, "text_emb_24h_nursing_clinicalbert.parquet")

In [ ]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"
device = "cuda" if torch.cuda.is_available() else "cpu"

notes = pd.read_parquet(NOTES_PATH)
notes["note_text_24h"] = notes["note_text_24h"].fillna("")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

@torch.no_grad()
def embed_text(text, max_length=512, stride=256):
    # tokenize without truncation to get full ids
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        add_special_tokens=True
    )
    input_ids = enc["input_ids"][0]  # (L,)
    L = input_ids.shape[0]

    # if short, single pass
    if L <= max_length:
        batch = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=False
        )
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch).last_hidden_state  # (1, T, 768)
        cls = out[:, 0, :].squeeze(0).cpu().numpy()  # (768,)
        return cls

    # long: sliding window over token ids
    cls_list = []
    start = 0
    while start < L:
        end = min(start + max_length, L)
        window_ids = input_ids[start:end]

        # ensure [CLS]/[SEP] exist by re-wrapping
        window_text = tokenizer.decode(window_ids, skip_special_tokens=True)
        batch = tokenizer(
            window_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=False
        )
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch).last_hidden_state
        cls = out[:, 0, :].squeeze(0).cpu().numpy()
        cls_list.append(cls)

        if end == L:
            break
        start += stride

    return np.mean(np.stack(cls_list, axis=0), axis=0)

# Run a small test batch first to verify speed and GPU memory usage
test_vec = embed_text(notes["note_text_24h"].iloc[0])
print("embedding dim:", test_vec.shape)

embeddings = []
for i, row in notes.iterrows():
    vec = embed_text(row["note_text_24h"])
    embeddings.append(vec)
    if (i+1) % 200 == 0:
        print("processed:", i+1)

emb = np.vstack(embeddings)  # (N, 768)
emb_df = pd.DataFrame(emb, columns=[f"text_emb_{i}" for i in range(emb.shape[1])])
out = pd.concat([notes[["icustay_id"]].reset_index(drop=True), emb_df], axis=1)

out.to_parquet(EMB_OUT, index=False, compression="snappy")
print("✅ saved:", EMB_OUT, out.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dim: (768,)


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

processed: 200
processed: 400
processed: 600
processed: 800
processed: 1000
processed: 1200
processed: 1400
processed: 1600
processed: 1800
processed: 2000
processed: 2200
processed: 2400
processed: 2600
processed: 2800
processed: 3000
processed: 3200
processed: 3400
processed: 3600
processed: 3800
processed: 4000
processed: 4200
processed: 4400
processed: 4600
processed: 4800
processed: 5000
processed: 5200
processed: 5400
processed: 5600
processed: 5800
processed: 6000
processed: 6200
processed: 6400
processed: 6600
processed: 6800
processed: 7000
processed: 7200
processed: 7400
processed: 7600
processed: 7800
processed: 8000
processed: 8200
processed: 8400
processed: 8600
processed: 8800
processed: 9000
processed: 9200
processed: 9400
processed: 9600
processed: 9800
processed: 10000
processed: 10200
processed: 10400
processed: 10600
processed: 10800
processed: 11000
processed: 11200
processed: 11400
processed: 11600
processed: 11800
processed: 12000
processed: 12200
processed: 12400